# Day 09 - CUDA Graph capture and replay

This lesson expands official
[Notebook 11](../notebooks/11_cuda_graphs.ipynb) into a stateful
correctness exercise. A tiny CuTe kernel increments one scalar.
Capturing `operations` launches creates one reusable graph; each
replay must add exactly `operations` to the scalar.

CUDA Graphs optimize repeated submission. They do not make the
increment instruction itself faster.

In [ ]:
import torch

import cutlass.cute as cute
from cutlass.cute.runtime import from_dlpack
from cuda.bindings.driver import CUstream
from torch.cuda import current_stream

## 1. Direct launch versus graph replay

Without a graph, Python submits every launch:

```text
Python -> launch 0 -> launch 1 -> ... -> launch N-1
```

With a graph, Python records the sequence once and later submits the
complete dependency graph:

```text
capture: Python -> [launch 0, launch 1, ..., launch N-1]
replay:  Python -> graph launch
```

This matters most for short kernels, where CPU dispatch latency is a
meaningful fraction of end-to-end time. Long compute-bound kernels
may show little difference.

## 2. Compile before stream capture

`cute.compile` may trace Python, generate MLIR/PTX, load a module,
and allocate runtime state. Those activities are not valid graph
nodes and may not be capture-safe. The required order is:

```text
construct representative arguments
    -> cute.compile(...)
    -> optional warmup
    -> begin graph capture
    -> call only the compiled runtime callable
    -> end capture
    -> replay
```

Calling the original `@cute.jit` function for the first time inside
capture can accidentally trigger compilation. Keep a separate
variable such as `compiled` and invoke that variable in the capture
region.

## 3. Bridge PyTorch streams to the CUDA driver

PyTorch owns the capture stream, while CuTe's launch accepts a
CUDA-driver stream handle:

```python
stream = CUstream(current_stream().cuda_stream)
```

Create the stream argument used for compilation before capture.
Inside `with torch.cuda.graph(graph):`, query `current_stream()`
again and wrap that active capture stream. Passing a different
stream means the launch may occur outside the captured graph or the
runtime may reject the mismatch.

In [ ]:
operations = 8
replays = 2
expected_after_each_replay = [
    operations * replay for replay in range(1, replays + 1)
]
assert expected_after_each_replay == [8, 16]
print(expected_after_each_replay)

## 4. Replay uses stable topology and addresses

A graph records concrete operations, dependencies, and argument
addresses. Replay does not rerun the Python loop and does not
automatically substitute newly allocated tensors.

Safe baseline:

- keep captured tensors alive;
- update values in those tensors in place;
- keep shapes and launch topology fixed;
- synchronize before reading host-visible results.

Advanced graph-update APIs can change some node parameters, but
this lesson intentionally uses fixed storage.

## 5. Challenge: capture repeated CuTe launches

Fill five TODOs:

1. Launch the kernel on the supplied `CUstream`.
2. Wrap PyTorch's current stream.
3. Compile before capture.
4. Capture `operations` compiled calls on the active graph stream.
5. Replay twice and validate cumulative state.

The first replay should produce `operations`; the second should
produce `2 * operations`.

In [ ]:
@cute.kernel
def increment_kernel(output: cute.Tensor):
    output[0] = output[0] + 1.0


@cute.jit
def increment(output: cute.Tensor, stream: CUstream):
    # TODO(1): launch increment_kernel into the supplied stream.
    raise NotImplementedError("Day 09 TODO(1): launch with stream=stream")


def run(operations: int) -> None:
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("Day 09 needs a CUDA device.")
    output = torch.zeros(1, device="cuda", dtype=torch.float32)
    output_dsl = from_dlpack(output)

    # TODO(2): wrap current_stream().cuda_stream in CUstream.
    # TODO(3): precompile increment with `output_dsl` and that stream.
    raise NotImplementedError("Day 09 TODO(2-3): stream and precompile")

    output.zero_()
    graph = torch.cuda.CUDAGraph()

    # TODO(4): use `with torch.cuda.graph(graph):` and enqueue `operations`
    # calls to the compiled function using the stream active during capture.

    output.zero_()
    # TODO(5): replay and assert output == operations, then replay again and
    # assert output == operations * 2. Synchronize before each assertion.

## 6. Correctness experiment

Run after filling all TODOs. Try one and many operations to verify
that capture topology determines the increment count.

In [ ]:
run(operations=1)
run(operations=8)

## 7. Measure submission overhead

After correctness passes, compare a direct Python loop with one
graph replay. CUDA events measure GPU-stream elapsed time; wall
time includes more Python overhead. For very small kernels, report
which timer you used.

```python
# Conceptual experiment:
start.record()
for _ in range(operations):
    compiled(output_dsl, stream)
end.record()

start.record()
graph.replay()
end.record()
```

Warm both paths before measuring. A graph can reduce gaps between
kernels even though each kernel's device instructions are unchanged.

## 8. Common capture failures

| Symptom | Likely cause |
| --- | --- |
| Capture error during first call | JIT compilation happened inside capture |
| Graph replays but output stays zero | Launch used a non-captured stream |
| First replay is too large | Recording work was not cleared before replay |
| Later tensor is unchanged | A new allocation replaced the captured address |
| Host reads stale output | Missing synchronization before assertion |

The capture itself executes work while recording. This lesson calls
`output.zero_()` after capture so replay accounting starts from zero.

## 9. Where graphs fit in real workloads

Graphs are useful when a stable sequence repeats: inference steps,
optimizer updates, communication/computation schedules, or a fixed
stack of small kernels. They compose with Day 08 benchmarking and
Day 10 GEMM, but solve a different bottleneck:

- kernel optimization reduces device execution time;
- CUDA Graphs reduce repeated host submission overhead.

**Checkpoint:** explain why changing tensor contents can be safe
while replacing the tensor allocation is not.